# Navionics GPX sailing-trip logbook (XKCD style)

Parse one or more **Navionics Boating App** GPX exports and produce, per trip:
a per-track summary table, a hand-drawn (**XKCD-style**) route map with a
simplified coastline, an optional interactive map, and per-track speed profiles.

Core parsing and statistics use only the Python standard library. The tables use
`pandas`, the static maps use `matplotlib` (+ `shapely` for the simplified
coastline), and the optional interactive map uses `folium`:

```
pip install pandas matplotlib shapely folium
```
(In this repo's Docker image these are already installed.)

**Data:** the Navionics GPX exports are large and **gitignored** — they are not
shipped with the repo. Place your exports in `work/datasets/` under the filenames
referenced in the `TRIPS` dict below. The simplified coastlines in
`coastlines/*.geojson` (derived from Natural Earth 1:10m) *are* committed.

**Usage:** configure your trips in the `TRIPS` dict below, then *Run All*.
Every trip renders in sequence — no per-trip re-editing needed.

## 0. Trip configuration

Each trip needs a GPX path. `harbours` (label → (lat, lon)) and `tz` (an IANA
timezone for local-time display) are optional but recommended. Harbour
positions here are **inferred from the track coordinates** — adjust them to
the places you actually stopped.

In [ ]:
TRIPS = {
    "Ionian Sea (Preveza / Lefkada), Jul 2025": {
        "gpx": "../datasets/navionics_archive_export_2025_preveza.gpx",
        "coast": "coastlines/preveza.geojson",
        "tz": "Europe/Athens",
        "harbours": {
            "Lefkada":  (38.832, 20.711),
            "Preveza":  (38.955, 20.755),
            "Nidri":    (38.675, 20.712),
            "Sivota":   (38.618, 20.688),
            "Meganisi": (38.667, 20.808),
            "Skorpios": (38.760, 20.802),
        },
    },
    "Sardinia-Corsica (Olbia / Bonifacio), Jun-Jul 2026": {
        "gpx": "../datasets/navionics_archive_export_2026_olbia.gpx",
        "coast": "coastlines/olbia.geojson",
        "tz": "Europe/Rome",
        "harbours": {
            "Olbia":          (40.926, 9.510),
            "Tavolara":       (40.834, 9.688),
            "La Maddalena":   (41.217, 9.481),
            "Porto-Vecchio":  (41.553, 9.320),
            "Bonifacio":      (41.388, 9.162),
            "Costa Smeralda": (41.072, 9.533),
        },
    },
}

## 1. Setup

All imports and tuning constants in one place. `folium` is optional — the notebook degrades gracefully without it (see section 5).

In [ ]:
import json
import math
import os
import statistics
from dataclasses import dataclass, field
from datetime import datetime
from xml.etree import ElementTree as ET
from zoneinfo import ZoneInfo

import pandas as pd
import matplotlib.pyplot as plt

try:
    import folium
    HAVE_FOLIUM = True
except ImportError:
    HAVE_FOLIUM = False

try:
    from shapely.geometry import shape
    HAVE_SHAPELY = True
except ImportError:
    HAVE_SHAPELY = False

GPX_NS = "http://www.topografix.com/GPX/1/1"
NS = {"g": GPX_NS}

MS_TO_KNOTS = 1.94384     # metres/second -> knots
KM_TO_NM = 1 / 1.852      # kilometres -> nautical miles (1 nm = 1.852 km)
MOVING_THRESHOLD_MS = 0.5 # position-derived pairwise speed below this = stationary
GLITCH_MAX_KNOTS = 40.0   # position-derived speeds above this are GPS spikes
ANCHORED_MAX_NM = 1.0     # tracks moving less than this are anchor watches
SPEED_SUSPECT_RATIO = 1.5 # Doppler avg vs implied avg disagreement => suspect

# --- rendering (XKCD static maps + speed profile) -------------------------
MAP_MAX_POINTS = 1500     # downsample each track to ~this before plotting; the
PROFILE_MAX_POINTS = 2000 # xkcd sketch filter is slow/mushy on 10k+ vertices
COAST_SIMPLIFY_TOLERANCE = 0.005  # deg; Douglas-Peucker on the coast so it wiggles
                                  # hand-drawn instead of smearing (0 = full detail)
MAP_MARGIN_DEG = 0.03     # padding around the track bounds for the map view

## 2. Data model and parsing

Navionics stores its own (Doppler) speed in a `navionics_speed` extension on
every point — normally more reliable than deriving speed from successive
positions, though occasionally a whole track's speed field is corrupt (see the
`speed_suspect` flag in section 3).

Tracks are sorted **chronologically** immediately after parsing; Navionics
exports are sometimes reverse-ordered, and everything downstream (tables, map
legends, `TRACK_INDEX`) assumes time order.

In [ ]:
@dataclass
class TrackPoint:
    lat: float
    lon: float
    time: datetime
    speed_ms: float           # navionics_speed (the app's own value)
    ele: float | None


@dataclass
class Track:
    name: str
    points: list = field(default_factory=list)


def _parse_time(text: str) -> datetime:
    # e.g. 2025-07-11T05:17:02.388Z
    return datetime.strptime(text.replace("Z", "+0000"), "%Y-%m-%dT%H:%M:%S.%f%z")


def parse_gpx(path: str) -> list:
    root = ET.parse(path).getroot()
    tracks = []
    for trk in root.findall("g:trk", NS):
        name = trk.findtext("g:name", default="(unnamed)", namespaces=NS)
        track = Track(name=name)
        for tp in trk.iter(f"{{{GPX_NS}}}trkpt"):
            speed = tp.findtext(".//g:navionics_speed", namespaces=NS)
            ele = tp.findtext("g:ele", namespaces=NS)
            track.points.append(TrackPoint(
                lat=float(tp.get("lat")),
                lon=float(tp.get("lon")),
                time=_parse_time(tp.findtext("g:time", namespaces=NS)),
                speed_ms=float(speed) if speed else 0.0,
                ele=float(ele) if ele else None,
            ))
        tracks.append(track)
    tracks.sort(key=lambda t: t.points[0].time)   # chronological, always
    return tracks


def haversine_m(lat1, lon1, lat2, lon2):
    """Great-circle distance in metres."""
    r = 6_371_000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    a = (math.sin(dphi / 2) ** 2
         + math.cos(p1) * math.cos(p2) * math.sin(dlmb / 2) ** 2)
    return 2 * r * math.asin(math.sqrt(a))

## 3. Per-track statistics

Distances come straight from the positions. "Moving" is decided from the
**positions** (pairwise speed > 0.5 m/s, GPS spikes excluded), never from the
Doppler field — so a broken speed sensor can't distort what counts as moving.
Speed is then reported two independent ways:

- **avg_moving_kt** — the app's Doppler speed averaged over the moving
  segments (time at anchor or on a beach stop doesn't dilute it).
- **implied_kt** — moving distance / moving time, purely position-derived.

If those two disagree by more than 50 %, **speed_suspect** is set and
`implied_kt` is the one to trust. When they agree, an unusual number is real:
in the Olbia data, Track 014 averages ~16 kt for an hour — both sensors
concur, so that's a genuine fast motor/RIB excursion, not a sensor fault.

- **max_kt** is the Doppler maximum over moving segments with per-track
  outlier rejection: values above 3× the track's median moving speed are
  treated as glitches. This keeps a genuine 34 kt peak on a fast motor day
  while rejecting an 80 kt spike on a 5 kt sailing day. Rejected values (and
  the position spike every Navionics track has right after its first fix)
  land in `glitch_kt`.

Each track is also classified: less than 1 nm of movement ⇒ **anchored**
(an overnight anchor watch, not a sailing leg).

In [ ]:
@dataclass
class TrackStats:
    name: str
    n_points: int
    start: datetime
    end: datetime
    duration_s: float
    distance_nm: float
    moving_nm: float
    moving_time_s: float
    avg_moving_knots: float
    implied_knots: float
    max_reported_knots: float
    max_realistic_knots: float
    glitch_knots: float | None
    kind: str                 # "sailing" | "anchored"
    speed_suspect: bool


def analyse(track) -> TrackStats:
    pts = track.points
    start, end = pts[0].time, pts[-1].time

    distance = moving_dist = moving_time = 0.0
    max_reported = max_realistic = 0.0
    glitch = None
    moving_doppler = []

    for prev, cur in zip(pts, pts[1:]):
        d = haversine_m(prev.lat, prev.lon, cur.lat, cur.lon)
        dt = (cur.time - prev.time).total_seconds()
        distance += d
        if dt <= 0:
            continue

        v_pos_ms = d / dt
        v_pos_kt = v_pos_ms * MS_TO_KNOTS
        if v_pos_kt > GLITCH_MAX_KNOTS:          # GPS spike: exclude entirely
            glitch = max(glitch or 0.0, v_pos_kt)
            continue
        max_realistic = max(max_realistic, v_pos_kt)

        if v_pos_ms > MOVING_THRESHOLD_MS:       # position-gated "moving"
            moving_dist += d
            moving_time += dt
            moving_doppler.append(cur.speed_ms)

    distance_nm = distance / 1000 * KM_TO_NM
    moving_nm = moving_dist / 1000 * KM_TO_NM
    avg_moving = (sum(moving_doppler) / len(moving_doppler) * MS_TO_KNOTS
                  if moving_doppler else 0.0)

    # Robust Doppler max: reject per-track outliers (glitches can lurk below
    # any global ceiling). Cap = 3x the track's median moving speed, floor
    # 15 kt so slow days can still show a genuine fast surf.
    if moving_doppler:
        med_kt = statistics.median(moving_doppler) * MS_TO_KNOTS
        cap_kt = max(3 * med_kt, 15.0)
        kept = [v * MS_TO_KNOTS for v in moving_doppler
                if v * MS_TO_KNOTS <= cap_kt]
        max_reported = (max(kept) / MS_TO_KNOTS) if kept else 0.0
        top = max(v * MS_TO_KNOTS for v in moving_doppler)
        if top > cap_kt:
            glitch = max(glitch or 0.0, top)
    implied = (moving_nm / (moving_time / 3600)) if moving_time > 0 else 0.0
    kind = "anchored" if moving_nm < ANCHORED_MAX_NM else "sailing"
    # two independent sensors: flag only when they disagree badly
    suspect = (kind == "sailing" and implied > 0
               and not (1 / SPEED_SUSPECT_RATIO
                        < avg_moving / implied
                        < SPEED_SUSPECT_RATIO))

    return TrackStats(
        name=track.name, n_points=len(pts), start=start, end=end,
        duration_s=(end - start).total_seconds(),
        distance_nm=distance_nm, moving_nm=moving_nm,
        moving_time_s=moving_time,
        avg_moving_knots=avg_moving, implied_knots=implied,
        max_reported_knots=max_reported * MS_TO_KNOTS,
        max_realistic_knots=max_realistic, glitch_knots=glitch,
        kind=kind, speed_suspect=suspect,
    )

## 4. Load and summarise — all trips

Loads every configured trip once into `trip_data` (used by all later cells),
then shows a summary table and totals per trip. Times are shown in the trip's
**local timezone**.

Reading the two tables: `kind` separates real sailing legs from overnight
anchor watches; a ⚠ in `spd?` would mean Doppler and position speeds disagree
badly for that track — read `implied_kt` in that case. A large `glitch_kt`
(like Montag's 83 kt in the Ionian data) means that track's speed sensor
emitted garbage at some point: its averages stay trustworthy (spikes barely
move a mean over thousands of points) but treat its `max_kt` with caution. (Note the Olbia
Track 014: ~16 kt average is *real* — an hour-long fast motor/RIB run where
both sensors agree, followed by a two-hour stop.)

In [ ]:
trip_data = {}   # trip name -> {"tracks": [...], "stats": [...], "cfg": {...}}

for trip_name, cfg in TRIPS.items():
    if not os.path.exists(cfg["gpx"]):
        raise FileNotFoundError(
            f"GPX for '{trip_name}' not found at {cfg['gpx']}. "
            "The Navionics exports are gitignored and not shipped with the repo — "
            "place your export at work/datasets/ under this filename "
            "(see the intro cell / README).")
    tracks = parse_gpx(cfg["gpx"])
    stats = [analyse(t) for t in tracks]
    trip_data[trip_name] = {"tracks": tracks, "stats": stats, "cfg": cfg}

for trip_name, td in trip_data.items():
    stats, cfg = td["stats"], td["cfg"]
    tz = ZoneInfo(cfg.get("tz", "UTC"))

    df = pd.DataFrame([{
        "track": s.name,
        "kind": s.kind,
        "start_local": s.start.astimezone(tz).strftime("%a %d %b %H:%M"),
        "points": s.n_points,
        "duration_h": round(s.duration_s / 3600, 1),
        "distance_nm": round(s.distance_nm, 1),
        "avg_moving_kt": round(s.avg_moving_knots, 1),
        "implied_kt": round(s.implied_knots, 1),
        "max_kt": round(s.max_reported_knots, 1),
        "glitch_kt": round(s.glitch_knots) if s.glitch_knots else None,
        "spd?": "⚠" if s.speed_suspect else "",
    } for s in stats])

    sailing = [s for s in stats if s.kind == "sailing"]
    print(f"═══ {trip_name} ═══")
    display(df)
    print(f"Sailing legs: {len(sailing)}   "
          f"anchor watches: {len(stats) - len(sailing)}")
    print(f"Total distance: {sum(s.distance_nm for s in stats):.1f} nm   "
          f"recorded time: {sum(s.duration_s for s in stats) / 3600:.1f} h   "
          f"points: {sum(s.n_points for s in stats):,}")
    lo = min(s.start for s in stats).astimezone(tz)
    hi = max(s.end for s in stats).astimezone(tz)
    print(f"Span: {lo:%a %d %b %Y %H:%M} -> {hi:%a %d %b %Y %H:%M} ({tz.key})")
    print()

## 5. Static route maps — all trips (XKCD style)

One hand-drawn figure per trip, rendered inside `plt.xkcd()`: a **simplified
coastline** (Natural Earth 1:10m, thinned by `COAST_SIMPLIFY_TOLERANCE` so it
sketches cleanly) sits behind the routes; sailing legs get a coloured line and
a legend entry; anchor watches are a single open circle (no legend clutter).
Each track is downsampled to ~`MAP_MAX_POINTS` first — the xkcd sketch filter
is slow and turns to mush on tens of thousands of vertices. The aspect ratio
uses an equirectangular correction so the coastline shape stays true. These
figures survive `nbconvert` to HTML/PDF, so they are the canonical maps for
export.

In [ ]:
def load_coast(path):
    """Simplified coastline as a list of (xs, ys) polylines, or [] if unavailable.

    Douglas-Peucker simplification keeps the vertex count low so plt.xkcd()'s
    sketch filter produces a hand-drawn coast instead of a fuzzy smear."""
    if not (HAVE_SHAPELY and path and os.path.exists(path)):
        return []
    geom = shape(json.load(open(path))["geometry"])
    if COAST_SIMPLIFY_TOLERANCE:
        geom = geom.simplify(COAST_SIMPLIFY_TOLERANCE, preserve_topology=True)
    parts = geom.geoms if geom.geom_type.startswith("Multi") else [geom]
    return [(list(g.xy[0]), list(g.xy[1])) for g in parts]


for trip_name, td in trip_data.items():
    tracks, stats, cfg = td["tracks"], td["stats"], td["cfg"]
    by_name = {s.name: s for s in stats}
    all_lats = [p.lat for t in tracks for p in t.points]
    all_lons = [p.lon for t in tracks for p in t.points]
    mean_lat = sum(all_lats) / len(all_lats)
    total_nm = sum(s.distance_nm for s in stats)
    coast = load_coast(cfg.get("coast"))

    with plt.xkcd():
        fig, ax = plt.subplots(figsize=(7, 7))
        for xs, ys in coast:                     # hand-drawn coastline, behind everything
            ax.plot(xs, ys, color="black", lw=0.9, zorder=0)
        cmap = plt.get_cmap("tab10")
        colour_idx = 0
        for t in tracks:
            s = by_name[t.name]
            if s.kind == "anchored":
                p0 = t.points[0]
                ax.plot(p0.lon, p0.lat, "o", mfc="none", mec="grey", ms=7, mew=1.4)
                continue
            step = max(1, len(t.points) // MAP_MAX_POINTS)   # thin for the sketch filter
            pts = t.points[::step]
            ax.plot([p.lon for p in pts], [p.lat for p in pts],
                    color=cmap(colour_idx % 10), lw=1.3,
                    label=f"{s.name}  {s.distance_nm:.1f} nm")
            colour_idx += 1

        for name, (la, lo) in cfg.get("harbours", {}).items():
            ax.plot(lo, la, "o", color="black", ms=4)
            ax.annotate(name, (lo, la), textcoords="offset points",
                        xytext=(5, 3), fontsize=8)

        # keep the view on the track; the coastline extends past it and is clipped
        ax.set_xlim(min(all_lons) - MAP_MARGIN_DEG, max(all_lons) + MAP_MARGIN_DEG)
        ax.set_ylim(min(all_lats) - MAP_MARGIN_DEG, max(all_lats) + MAP_MARGIN_DEG)
        ax.set_aspect(1 / math.cos(math.radians(mean_lat)))
        ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
        ax.set_title(f"{trip_name} — {total_nm:.1f} nm total\n"
                     f"(open grey circles = overnight anchorages)", fontsize=10)
        ax.legend(fontsize=7, loc="best")
        ax.grid(alpha=0.3)
        fig.tight_layout()
        plt.show()

## 6. Interactive map (optional)

A Leaflet map on a real basemap for **one** trip at a time — set
`INTERACTIVE_TRIP` to a key of `TRIPS` (or leave as-is for the first trip).
Requires `folium`; the cell explains itself and moves on if it's missing.

Note: this cell embeds ~350 KB of Leaflet into the notebook and does **not**
render in PDF exports. Clear its output before committing / exporting if size
matters — the static maps in section 5 are the export-safe versions.

In [ ]:
INTERACTIVE_TRIP = list(TRIPS)[1]   # or name a trip explicitly

if not HAVE_FOLIUM:
    print("folium not installed -- skipping interactive map. "
          "Run: pip install folium")
else:
    td = trip_data[INTERACTIVE_TRIP]
    tracks, stats = td["tracks"], td["stats"]
    by_name = {s.name: s for s in stats}
    lats = [p.lat for t in tracks for p in t.points]
    lons = [p.lon for t in tracks for p in t.points]

    fmap = folium.Map(tiles="OpenStreetMap")
    fmap.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])

    palette = ["#e6194B", "#f58231", "#d4b800", "#3cb44b",
               "#1aa3c4", "#4363d8", "#911eb4", "#f032e6"]
    colour_idx = 0
    for t in tracks:
        s = by_name[t.name]
        if s.kind == "anchored":
            p0 = t.points[0]
            folium.CircleMarker((p0.lat, p0.lon), radius=6, color="grey",
                                fill=False,
                                tooltip=f"{t.name} (anchorage)").add_to(fmap)
            continue
        step = max(1, len(t.points) // 1500)
        coords = [(p.lat, p.lon) for p in t.points[::step]]
        folium.PolyLine(coords, color=palette[colour_idx % len(palette)],
                        weight=3, opacity=0.85,
                        tooltip=f"{t.name}  {s.distance_nm:.1f} nm").add_to(fmap)
        folium.CircleMarker(coords[0], radius=4, color="black", fill=True,
                            tooltip=f"{t.name} start").add_to(fmap)
        colour_idx += 1

    display(fmap)

## 7. Speed profile for one track

Doppler speed over the course of one day — stops, motoring and sailing legs
show up clearly. Pick the trip and the track (by name, which is unambiguous;
indexes shift between datasets). If the chosen track is `speed_suspect`,
the title says so.

In [ ]:
PROFILE_TRIP = list(TRIPS)[0]      # or name a trip explicitly
PROFILE_TRACK = None               # None = first sailing track, or e.g. "Montag"

td = trip_data[PROFILE_TRIP]
by_name = {s.name: s for s in td["stats"]}
if PROFILE_TRACK is None:
    PROFILE_TRACK = next(s.name for s in td["stats"] if s.kind == "sailing")
t = next(tr for tr in td["tracks"] if tr.name == PROFILE_TRACK)
s = by_name[PROFILE_TRACK]

step = max(1, len(t.points) // PROFILE_MAX_POINTS)   # thin for the sketch filter
pts = t.points[::step]
t0 = pts[0].time
elapsed_h = [(p.time - t0).total_seconds() / 3600 for p in pts]
speed_kt = [p.speed_ms * MS_TO_KNOTS for p in pts]

# data-driven y-limit: keep genuine fast legs (Olbia Track 014 ~16 kt) visible
# while still clipping the residual Doppler spikes above the robust max
y_top = min(max(6.0, s.max_reported_knots * 1.2), 40.0)
suffix = (f"  (speed sensor unreliable — trust implied avg {s.implied_knots:.1f} kt)"
          if s.speed_suspect else "")
with plt.xkcd():
    plt.figure(figsize=(10, 3))
    plt.plot(elapsed_h, speed_kt, lw=0.9)
    plt.title(f"How fast did we go? — {t.name}, {PROFILE_TRIP}{suffix}", fontsize=10)
    plt.xlabel("hours underway")
    plt.ylabel("speed (knots)")
    plt.ylim(0, y_top)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

---
*Methodology notes: distances and durations come straight from the track
points; averages use only points moving faster than 0.3 m/s so anchor time
doesn't dilute them; `implied_kt` (distance over moving time) is the
sensor-independent cross-check. All track names come from the GPX; harbour
labels are user-supplied inferences, not data in the file. Every Navionics
track carries a 1–2 s GPS spike right after the fix — these are filtered from
maxima and reported per-track as `glitch_kt` in `analyse()`.*